<img src="https://www.unir.net/wp-content/uploads/2019/11/Unir_2021_logo.svg" width="240" height="240" align="right"/>

<center><h1>Trabajo de final de máster</header1></center>

# Importación de librerias necesarias

In [ ]:
#Para esta actividad se importarán las siguientes librerías:
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

####### Preprocesamiento

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

####### Regresión lineal

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


## Cargar el Dataset

In [ ]:
students_data = pd.read_csv('Data/Teachers_con_datos_PISA.csv', sep=';', low_memory=False)
students_data.isna().sum()

students_data = students_data.dropna(axis=1)

students_data.head()

## Caracterización del Dataset

In [ ]:
#Código que responde a la descripción anterior

#Imprimimos la longitud de valores del dataFrame, obteniendo el número total de instancias
print("Número de instancias en total:", len(students_data.values))

#Imprimimos el total de atributos de entrada (columnas), y restamos uno que corresponde a la variable objetivo
print("Número de atributos de entrada:", len(students_data.columns) -2)
print("Tipos de atributo:", students_data.dtypes)
students_data.info(verbose=True)
print("Analisis EDA del dataset")
print(students_data.describe())

##### Gráficos

classNames = set()
#Recorremos los valores de la columna "class"
for x in students_data["COUNTRY"].values:
    #Añadimos el valor al set
    classNames.add(x)

#Imprimimos la longitud del set de classnames, que solo contendra el nombre de cada clase una unica vez
print("Número de países:", len(classNames))
classCount  = {}
#Recorremos el set de los nombres de clase
for className in classNames:
    counter = 0
    #Recorremos los valores de la columna "class"
    for x in students_data["COUNTRY"]:
        #En caso de coincidir el className actual con el valor dado incrementamos el contador
        if(x == className):
            counter += 1
    #Guardamos en un diccionario para mostrar después el gráfico
    classCount[className] = counter
    print("Para el país", className, "existen:", counter, "instancias")
classCount = dict(sorted(classCount.items()))
plt.figure(figsize=(15, 9))
plt.bar(classCount.keys(), classCount.values())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Preprocesamiento del dataset. Transformaciones previas necesarias para la modelación

In [ ]:
students_data.drop(['SCHOOLID','CLASSID','TEID','WAVE', 'FSTWT', 'TSC_7477', 'PISA_Math', 'PISA_Science'], axis = 1, inplace=True)

if students_data['PISA_Media'].dtype == 'str':
    students_data['PISA_Media'] = pd.to_numeric(students_data['PISA_Media'].str.replace(',', '.'), errors='coerce')

numeric_cols = students_data.select_dtypes(include=[np.number]).columns

# Reemplazamos 99X por NaN y luego por la mediana
students_data[numeric_cols] = students_data[numeric_cols].replace(990, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(991, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(992, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(993, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(994, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(995, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(996, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(997, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(998, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].replace(999, np.nan)
students_data[numeric_cols] = students_data[numeric_cols].fillna(students_data[numeric_cols].median())

columnsToEncode = list(students_data.select_dtypes(include=['str']))
le = LabelEncoder()
for feature in columnsToEncode:
    try:
        students_data[feature] = le.fit_transform(students_data[feature])
    except:
        print('Error encoding '+ feature)


students_data.head()

students_data.to_csv("CacheTeachers/teachers_dropna_encoded.csv", sep=";", index = False)

## División del dataset en datos de entrenamiento y datos de test

In [ ]:
#Código que realice la división en entrenamiento y test, de acuerdo con la estretgia de evaluación planeada. Describa cuál es.
intro_attr = students_data

#Separamos los bloques de test (80%) de los bloques de entrenamiento (20%)
train_dataset = intro_attr.sample(frac=0.8,random_state=0)
test_dataset = intro_attr.drop(train_dataset.index)

# Calculamos las estaísitcas para poder normalizar posteriormente
train_stats = train_dataset.describe()
train_stats.pop("PISA_Media")
train_stats = train_stats.transpose()

# Extraemos la variable objetivo del conjunto
train_labels = train_dataset.pop('PISA_Media')
test_labels = test_dataset.pop('PISA_Media')

# Normalización
def norm(x):
  return (x - train_stats['mean']) / (train_stats['std'] - 1e-9)

normed_train_data = norm(train_dataset)
normed_test_data = norm(test_dataset)

train_stats.to_csv("CacheTeachers/train_stats.csv")

## Propuesta de arquitectura de red neuronal

Describe:

+ las neuronas en la capa de entrada
+ las capas intermedias – al menos dos –
+ capa de salida
+ funciones de activación

In [ ]:
from pathlib import Path

students_model = Path("CacheTeachers/teachers.keras")    

In [ ]:
# Código de la estructuración de la red
# - La capa oculta 1 tiene más neuronas que la capa de entrada para permitir la extracción de características complejas.
# - La capa oculta 2 tiene menos neuronas que la capa oculta 1 para reducir la dimensionalidad y evitar el sobreajuste.
# - La capa de salida tiene 1 neurona, ya que solo hay una variable de salida.

dataset_len = len(train_dataset.keys())
expansion_val = 2

def build_model():
    inputs = keras.Input(shape=(dataset_len,))
    mid = layers.Dense(int(dataset_len*expansion_val), activation='relu')(inputs)
    mid = layers.Dense(int(dataset_len/expansion_val), activation='relu')(mid)
    outputs = layers.Dense(1)(mid)

    model = keras.Model(inputs=inputs, outputs=outputs)

    optimizer = tf.keras.optimizers.RMSprop(0.001)

    model.compile(loss='mse',
                optimizer=optimizer,
                metrics=['mae', 'mse'])
    return model

if students_model.exists():
    model = keras.models.load_model("CacheTeachers/teachers.keras")
else:
    model = build_model()

In [ ]:
#Código de la inspección del modelo de red
model.summary()

##  Ajuste de modelo de Regresión RNA

In [ ]:
import json

if not students_model.exists():
    #Código de ajuste y entrenamiento
    class PrintDot(keras.callbacks.Callback):
      def on_epoch_end(self, epoch, logs):
        if epoch % 100 == 0: print('')
        print('.', end='')
    
    EPOCHS = 1000
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, min_delta=0.001)
    
    history = model.fit(
      normed_train_data, train_labels,
      epochs=EPOCHS, validation_split = 0.2, verbose=0,
      callbacks=[PrintDot(), early_stopping])
    
    model.save("CacheTeachers/teachers.keras")
    with open("CacheTeachers/training_history.json", "w") as f:
        json.dump(history.history, f)
    with open("CacheTeachers/training_epoch.json", "w") as f:
        json.dump(history.epoch, f)

In [ ]:
with open("CacheTeachers/training_history.json", "r") as f:
    history = json.load(f)
with open("CacheTeachers/training_epoch.json", "r") as f:
    history_epoch = json.load(f)

## Evaluación de modelo RNA

In [ ]:
hist = history
hist['epoch'] = history_epoch
# print(hist.tail())

def plot_history(history, history_epoch):
  hist = history
  hist['epoch'] = history_epoch

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Abs Error [PISA_Media]')
  plt.plot(hist['epoch'], hist['mae'],
           label='Train Error')
  plt.plot(hist['epoch'], hist['val_mae'],
           label = 'Val Error')
  plt.ylim([0,1000])
  plt.legend()

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Square Error [$PISA_Media^2$]')
  plt.plot(hist['epoch'], hist['mse'],
           label='Train Error')
  plt.plot(hist['epoch'], hist['val_mse'],
           label = 'Val Error')
  plt.ylim([0,1000])
  plt.legend()
  plt.show()


plot_history(history, history_epoch)

### Resultados para el conjunto de test.

In [ ]:
#Código de evaluación de la red propuesta (evaluación conjunto de test)
test_predictions = model.predict(normed_test_data).flatten()

plt.scatter(test_labels, test_predictions)
plt.xlabel('True Values [PISA_Media]')
plt.ylabel('Predictions [PISA_Media]')
plt.axis('equal')
plt.axis('square')
plt.xlim([400,600])
plt.ylim([400,600])
_ = plt.plot([0, 1000], [0, 1000])



In [ ]:
#Código de errores
error = test_predictions - test_labels
plt.hist(error, bins = 25)
plt.xlabel("Prediction Error [PISA_Media]")
_ = plt.ylabel("Count")

### Análisis de la Importancia de las Variables con Permutation Importance (sklearn)

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np

r = permutation_importance(model,
                           normed_test_data,
                           test_labels,
                           scoring='neg_mean_absolute_error')

# Organizar los resultados
sorted_idx = r.importances_mean.argsort()

# Obtener los nombres de las columnas en el orden correcto
feature_names = normed_test_data.columns[sorted_idx]

# Imprimir las importancias medias y sus desviaciones estándar
print("Importancia de las características (Mean Absolute Error):")
for i in sorted_idx[::-1]: # Imprimir de mayor a menor importancia
    print(f"{normed_test_data.columns[i]:<30} "
          f"{r.importances_mean[i]:.3f} "
          f"+/- {r.importances_std[i]:.3f}")

# Visualizar los resultados
fig, ax = plt.subplots(figsize=(10, 25))
ax.boxplot(r.importances[sorted_idx].T,
           vert=False, labels=feature_names)
ax.set_title("Importancia de las características por permutación (MAE)")
ax.set_xlabel("Disminución en MAE (cuanto más alto, más importante)")
ax.set_ylabel("Característica")
plt.tight_layout()
plt.show()